# RQ1 (fixed)

As per module leads email - "If the original code or dataset causes problems, document the issue, make any necessary justified adjustments, and follow the contingency provided." 

The following are the fixes needed for the code to produce real results.\
Every changed line in the code cells is marked with `# FIX n` and the original line is kept as a comment beneath `# ORIGINAL:`.

**Changes made**

- **FIX 1: train and score on the real labels.** The original used predicted_labels as both training and test labels. The kNN classifier was scored against another kNN's predictions rather than the ground truth, which produced 1.0 scores. \
The fix uses program_labels, the real labels taken from the folders.

- **FIX 2: use stratified 5-fold cross-validation instead of one 80/20 split.** A 20% test split of 22 or 30 files leaves only 5 or 6 test files, so a single error moves F1 by about 0.2. The split also wasn't stratified, so class balance in the test set was down to chance. \
The fix uses StratifiedKFold with cross_val_predict and the same KNeighborsClassifier(n_neighbors=3), so every file is tested exactly once.

- **FIX 3: skip files that appear in both classes.** Factory Method's factorymethod (3).java is identical to nonfm (11).java, and Singleton's singleton (3).java is identical to nons (14).java. \
Each of these is labelled both 1 and 0. When loading, a non-pattern file is now skipped if its content matches a pattern file.

- **FIX 4: load files in sorted order.** The original read each folder with os.listdir(), whose order depends on the operating system. \
The fix sorts the listing with the notebook's existing _natural_key, so their order is fixed.

In [1]:
import os
import re
import csv
import random
import shutil
from datetime import datetime
import torch
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import RobertaTokenizer, RobertaModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_val_predict  # FIX 2
from collections import Counter  # moved here so every cell can use it
from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(2137)

/Users/emmao/CS4928-Project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


How well can RoBERTa identify three Gang of Four patterns (Factory Method, Prototype, Singleton) in Java source code across 30 examples (15 true, 15 false) each?

In [2]:
# Where the frozen RQ1 samples are written (relative to this notebooks/ folder)
RQ1_ROOT = "../datasets/rq1_datasets"

# Positives are the keyword-named copies in <Pattern>/name/ (unambiguous IDs);
# negatives come from each pattern's non-* folder.
RQ1_SOURCES = {
    "factory_method": {
        "true":  "../datasets/Factory method/name",             # 11 *.java
        "false": "../datasets/Factory method/nonfactorymethod", # 82 *.java
    },
    "prototype": {
        "true":  "../datasets/Prototype/name",                  # 32 *.java
        "false": "../datasets/Prototype/nonprototype",          # 61 *.java
    },
    "singleton": {
        "true":  "../datasets/Singleton/name",                  # 25 *.java
        "false": "../datasets/Singleton/nonsingleton",          # 68 *.java
    },
}


def _natural_key(name):
    """Sort 'foo (2).java' before 'foo (10).java'."""
    return [int(part) if part.isdigit() else part.lower()
            for part in re.split(r"(\d+)", name)]


def build_rq1_dataset(pattern, n_true=15, n_false=None, seed=None):
    """Copy one frozen RQ1 sample into ../datasets/rq1_datasets/<pattern>/.

    pattern : "factory_method" | "prototype" | "singleton"
    n_true  : number of TRUE examples (capped at what the dataset provides)
    n_false : number of FALSE examples; None -> same as the actual n_true (1:1)
    seed    : None -> first N files in stable (natural) sorted order
              int  -> reproducible random sample using that seed

    Creates <pattern>/pattern/ (label 1) and <pattern>/non-pattern/ (label 0),
    rebuilds both folders on every call, and writes <pattern>/selection.csv
    with the frozen file IDs.
    """
    if pattern not in RQ1_SOURCES:
        raise ValueError(f"Unknown pattern {pattern!r}; choose from {list(RQ1_SOURCES)}")

    src_true  = RQ1_SOURCES[pattern]["true"]
    src_false = RQ1_SOURCES[pattern]["false"]

    true_files  = sorted((f for f in os.listdir(src_true)  if f.endswith(".java")), key=_natural_key)
    false_files = sorted((f for f in os.listdir(src_false) if f.endswith(".java")), key=_natural_key)

    # "max per class": never copy more than the dataset actually has
    n_true_sel  = min(n_true, len(true_files))
    n_false_sel = min(n_true_sel if n_false is None else n_false, len(false_files))

    if n_true_sel < n_true:
        print(f"  note: {pattern} has only {len(true_files)} true examples; using all of them.")
    if n_false is not None and n_false_sel < n_false:
        print(f"  note: {pattern} has only {len(false_files)} false examples; using all of them.")

    # pick the sample
    if seed is None:
        sel_true, sel_false = true_files[:n_true_sel], false_files[:n_false_sel]
        rule = "first N files in stable sorted order"
    else:
        rng = random.Random(seed)
        sel_true  = rng.sample(true_files,  n_true_sel)
        sel_false = rng.sample(false_files, n_false_sel)
        rule = f"random sample, seed={seed}"

    # rebuild the destination so it contains exactly this selection
    target    = os.path.join(RQ1_ROOT, pattern)
    dst_true  = os.path.join(target, "pattern")
    dst_false = os.path.join(target, "non-pattern")
    if os.path.exists(target):
        shutil.rmtree(target)
    os.makedirs(dst_true)
    os.makedirs(dst_false)

    for f in sel_true:
        shutil.copy2(os.path.join(src_true, f), os.path.join(dst_true, f))
    for f in sel_false:
        shutil.copy2(os.path.join(src_false, f), os.path.join(dst_false, f))

    # freeze the choice on disk (evidence for the report)
    with open(os.path.join(target, "selection.csv"), "w", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        writer.writerow(["class", "label", "filename", "source_folder"])
        writer.writerows([["pattern", 1, f, src_true] for f in sel_true])
        writer.writerows([["non-pattern", 0, f, src_false] for f in sel_false])
        writer.writerow([])
        writer.writerow([f"selection rule: {rule}"])
        writer.writerow([f"created: {datetime.now().isoformat(timespec='seconds')}"])

    print(f"{pattern}: {n_true_sel} true + {n_false_sel} false -> {target}")
    print(f'  java_programs_folder = "{target}"')
    print(f'  class_folders = {{"non-pattern": 0, "pattern": 1}}')

    return {"pattern": pattern, "n_true": n_true_sel, "n_false": n_false_sel, "folder": target}


def build_all_rq1(n_true=15, n_false=None, seed=None):
    """Build all three RQ1 patterns with the same true/false settings."""
    results = [build_rq1_dataset(p, n_true=n_true, n_false=n_false, seed=seed)
               for p in RQ1_SOURCES]
    print("\nSummary")
    for r in results:
        print(f"  {r['pattern']:15s} {r['n_true']:2d} true + {r['n_false']:2d} false "
              f"({r['n_true'] + r['n_false']:2d} files) -> {r['folder']}")
    return results

In [3]:
build_all_rq1(n_true=15, n_false=None)

  note: factory_method has only 11 true examples; using all of them.
factory_method: 11 true + 11 false -> ../datasets/rq1_datasets/factory_method
  java_programs_folder = "../datasets/rq1_datasets/factory_method"
  class_folders = {"non-pattern": 0, "pattern": 1}
prototype: 15 true + 15 false -> ../datasets/rq1_datasets/prototype
  java_programs_folder = "../datasets/rq1_datasets/prototype"
  class_folders = {"non-pattern": 0, "pattern": 1}
singleton: 15 true + 15 false -> ../datasets/rq1_datasets/singleton
  java_programs_folder = "../datasets/rq1_datasets/singleton"
  class_folders = {"non-pattern": 0, "pattern": 1}

Summary
  factory_method  11 true + 11 false (22 files) -> ../datasets/rq1_datasets/factory_method
  prototype       15 true + 15 false (30 files) -> ../datasets/rq1_datasets/prototype
  singleton       15 true + 15 false (30 files) -> ../datasets/rq1_datasets/singleton


[{'pattern': 'factory_method',
  'n_true': 11,
  'n_false': 11,
  'folder': '../datasets/rq1_datasets/factory_method'},
 {'pattern': 'prototype',
  'n_true': 15,
  'n_false': 15,
  'folder': '../datasets/rq1_datasets/prototype'},
 {'pattern': 'singleton',
  'n_true': 15,
  'n_false': 15,
  'folder': '../datasets/rq1_datasets/singleton'}]

# Factory Method Results

Expected Results:

Precision: 0.87
Recall: 0.77
F1-score: 0.78

Actual Results (stratified 5-fold CV, weighted average):

Precision: 0.90
Recall: 0.90
F1-score: 0.90

Sample: 11 true + 10 false (21 files; `nonfm (11).java` removed by FIX 3)

In [4]:
# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/rq1_datasets/factory_method" 

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

# FIX 3: read the pattern files first so a non-pattern file with identical content
# (same file labelled both 1 and 0) can be skipped.
pattern_folder = os.path.join(java_programs_folder, "pattern")
pattern_texts = set()
for program_file in os.listdir(pattern_folder):
    with open(os.path.join(pattern_folder, program_file), "r", encoding="iso-8859-1") as f:
        pattern_texts.add(f.read())

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    # FIX 4: sort so the file order (and therefore the CV folds) is the same on every OS
    # ORIGINAL: for program_file in os.listdir(folder_path):
    for program_file in sorted(os.listdir(folder_path), key=_natural_key):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                # FIX 3: skip label-conflicting duplicates
                # ORIGINAL: java_programs.append(f.read())
                program_text = f.read()
            if label == 0 and program_text in pattern_texts:
                print(f"FIX 3: skipping {program_file}, identical to a file in pattern/ (label conflict)")
                continue
            java_programs.append(program_text)

            program_labels.append(label)
            program_files.append(program_file)


from collections import Counter
print(Counter(program_labels))

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# FIX 1: the block below created pseudo-labels with a leave-one-out kNN vote and then
# used them as the "true" labels for training and testing. It is no longer used; the
# real labels (program_labels) are used instead.
# ORIGINAL:
# k = 5  # Number of neighbors to consider
# knn_indices = np.argsort(distance_matrix)[:, 1:k+1]
# predicted_labels = []
# for indices in knn_indices:
#     neighbor_labels = [program_labels[i] for i in indices]
#     predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
#     predicted_labels.append(predicted_label)
# predicted_labels = np.array(predicted_labels)

# FIX 2: one unstratified 80/20 split (5-6 test files) replaced with stratified 5-fold
# cross-validation, so every file is predicted exactly once by a model that did not see it.
# ORIGINAL:
# X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
#     normalized_embeddings,
#     predicted_labels,
#     program_files,
#     test_size=0.2,
#     random_state=0
# )
# knn.fit(X_train, y_train)
# predictions = knn.predict(X_test)
y_test = np.array(program_labels)  # FIX 1: real labels
files_test = program_files

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2137)  # FIX 2
predictions = cross_val_predict(knn, normalized_embeddings, y_test, cv=cv)  # FIX 2

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 28444.28it/s]


[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FIX 3: skipping nonfm (11).java, identical to a file in pattern/ (label conflict)
Counter({1: 11, 0: 10})



Generating embeddings:   0%|          | 0/21 [00:00<?, ?it/s]

...


Generating embeddings:  90%|█████████ | 19/21 [01:48<00:06,  3.31s/it]


Generating embeddings:  95%|█████████▌| 20/21 [01:50<00:03,  3.07s/it]


Generating embeddings: 100%|██████████| 21/21 [01:52<00:00,  2.55s/it]


Generating embeddings: 100%|██████████| 21/21 [01:52<00:00,  5.33s/it]

File: nonfm (1).java, Predicted Label: 0, True Label: 0
File: nonfm (2).java, Predicted Label: 0, True Label: 0
File: nonfm (3).java, Predicted Label: 0, True Label: 0
File: nonfm (4).java, Predicted Label: 0, True Label: 0
File: nonfm (5).java, Predicted Label: 0, True Label: 0
File: nonfm (6).java, Predicted Label: 0, True Label: 0
File: nonfm (7).java, Predicted Label: 0, True Label: 0
File: nonfm (8).java, Predicted Label: 0, True Label: 0
File: nonfm (9).java, Predicted Label: 1, True Label: 0
File: nonfm (10).java, Predicted Label: 0, True Label: 0
File: factorymethod (1).java, Predicted Label: 1, True Label: 1
File: factorymethod (2).java, Predicted Label: 1, True Label: 1
File: factorymethod (3).java, Predicted Label: 1, True Label: 1
File: factorymethod (4).java, Predicted Label: 1, True Label: 1
File: factorymethod (5).java, Predicted Label: 1, True Label: 1
File: factorymethod (6).java, Predicted Label: 1, True Label: 1
File: factorymethod (7).java, Predicted Label: 0, True 

# Prototype Results

Expected Results:

Precision: 0.88
Recall: 0.84
F1-score: 0.84

Actual Results (stratified 5-fold CV, weighted average):

Precision: 0.94
Recall: 0.93
F1-score: 0.93

Sample: 15 true + 15 false (30 files)

In [5]:
# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/rq1_datasets/prototype"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

# FIX 3: read the pattern files first so a non-pattern file with identical content
# (same file labelled both 1 and 0) can be skipped.
pattern_folder = os.path.join(java_programs_folder, "pattern")
pattern_texts = set()
for program_file in os.listdir(pattern_folder):
    with open(os.path.join(pattern_folder, program_file), "r", encoding="iso-8859-1") as f:
        pattern_texts.add(f.read())

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    # FIX 4: sort so the file order (and therefore the CV folds) is the same on every OS
    # ORIGINAL: for program_file in os.listdir(folder_path):
    for program_file in sorted(os.listdir(folder_path), key=_natural_key):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                # FIX 3: skip label-conflicting duplicates
                # ORIGINAL: java_programs.append(f.read())
                program_text = f.read()
            if label == 0 and program_text in pattern_texts:
                print(f"FIX 3: skipping {program_file}, identical to a file in pattern/ (label conflict)")
                continue
            java_programs.append(program_text)

            program_labels.append(label)
            program_files.append(program_file)


from collections import Counter
print(Counter(program_labels))

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# FIX 1: the block below created pseudo-labels with a leave-one-out kNN vote and then
# used them as the "true" labels for training and testing. It is no longer used; the
# real labels (program_labels) are used instead.
# ORIGINAL:
# k = 5  # Number of neighbors to consider
# knn_indices = np.argsort(distance_matrix)[:, 1:k+1]
# predicted_labels = []
# for indices in knn_indices:
#     neighbor_labels = [program_labels[i] for i in indices]
#     predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
#     predicted_labels.append(predicted_label)
# predicted_labels = np.array(predicted_labels)

# FIX 2: one unstratified 80/20 split (5-6 test files) replaced with stratified 5-fold
# cross-validation, so every file is predicted exactly once by a model that did not see it.
# ORIGINAL:
# X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
#     normalized_embeddings,
#     predicted_labels,
#     program_files,
#     test_size=0.2,
#     random_state=0
# )
# knn.fit(X_train, y_train)
# predictions = knn.predict(X_test)
y_test = np.array(program_labels)  # FIX 1: real labels
files_test = program_files

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2137)  # FIX 2
predictions = cross_val_predict(knn, normalized_embeddings, y_test, cv=cv)  # FIX 2

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14803.07it/s]


[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Counter({0: 15, 1: 15})



Generating embeddings:   0%|          | 0/30 [00:00<?, ?it/s]

...


Generating embeddings:  93%|█████████▎| 28/30 [00:46<00:03,  1.77s/it]


Generating embeddings:  97%|█████████▋| 29/30 [00:46<00:01,  1.38s/it]


Generating embeddings: 100%|██████████| 30/30 [00:47<00:00,  1.15s/it]


Generating embeddings: 100%|██████████| 30/30 [00:47<00:00,  1.58s/it]

File: nonp (1).java, Predicted Label: 0, True Label: 0
File: nonp (2).java, Predicted Label: 0, True Label: 0
File: nonp (3).java, Predicted Label: 0, True Label: 0
File: nonp (4).java, Predicted Label: 0, True Label: 0
File: nonp (5).java, Predicted Label: 0, True Label: 0
File: nonp (6).java, Predicted Label: 0, True Label: 0
File: nonp (7).java, Predicted Label: 0, True Label: 0
File: nonp (8).java, Predicted Label: 0, True Label: 0
File: nonp (9).java, Predicted Label: 0, True Label: 0
File: nonp (10).java, Predicted Label: 0, True Label: 0
File: nonp (11).java, Predicted Label: 0, True Label: 0
File: nonp (12).java, Predicted Label: 0, True Label: 0
File: nonp (13).java, Predicted Label: 0, True Label: 0
File: nonp (14).java, Predicted Label: 0, True Label: 0
File: nonp (15).java, Predicted Label: 0, True Label: 0
File: prototype (1).java, Predicted Label: 1, True Label: 1
File: prototype (2).java, Predicted Label: 0, True Label: 1
File: prototype (3).java, Predicted Label: 1, Tru

# Singleton Results

Expected Results:

Precision: 0.92
Recall: 0.9
F1-score: 0.91

Actual Results (stratified 5-fold CV, weighted average):

Precision: 0.86
Recall: 0.86
F1-score: 0.86

Sample: 15 true + 14 false (29 files; `nons (14).java` removed by FIX 3)

In [6]:
# Define the path to the folder containing your Java programs
java_programs_folder = "../datasets/rq1_datasets/singleton"

# Initialize the RoBERTa tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
model = RobertaModel.from_pretrained("roberta-base").to(device).eval()

# Function to extract embeddings from text
def get_embeddings(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).cpu().numpy()

# Function to calculate the distance matrix based on the chosen metric
def calculate_distance_matrix(embeddings, metric='cosine'):
    if metric == 'cosine':
        return cosine_distances(embeddings)
    elif metric == 'euclidean':
        return euclidean_distances(embeddings)
    else:
        raise ValueError(f"Unrecognized metric: {metric}")

# Switch between 'cosine' and 'euclidean'
distance_metric = 'euclidean'  # Change to 'euclidean' for Euclidean distance

# Initialize lists to store Java programs and their corresponding labels
java_programs = []
program_labels = []
program_files = []
# # Load Java programs from the folder and classify them as positive or negative
# for program_file in os.listdir(java_programs_folder):
#     file_path = os.path.join(java_programs_folder, program_file)

#     # Check if the item is a file, not a directory
#     if os.path.isfile(file_path):
#         with open(file_path, 'r', encoding='iso-8859-1') as f:
#             program_text = f.read()
#             java_programs.append(program_text)

#             # Check if the keyword "singleton" is present in the filename
#             if "singleton" in program_file:
#                 program_labels.append(1)  # Positive class (implements singleton)
#             else:
#                 program_labels.append(0)  # Negative class (does not implement singleton)


class_folders = {"non-pattern": 0, "pattern": 1}

# FIX 3: read the pattern files first so a non-pattern file with identical content
# (same file labelled both 1 and 0) can be skipped.
pattern_folder = os.path.join(java_programs_folder, "pattern")
pattern_texts = set()
for program_file in os.listdir(pattern_folder):
    with open(os.path.join(pattern_folder, program_file), "r", encoding="iso-8859-1") as f:
        pattern_texts.add(f.read())

for folder_name, label in class_folders.items():
    folder_path = os.path.join(java_programs_folder, folder_name)

    # FIX 4: sort so the file order (and therefore the CV folds) is the same on every OS
    # ORIGINAL: for program_file in os.listdir(folder_path):
    for program_file in sorted(os.listdir(folder_path), key=_natural_key):
        file_path = os.path.join(folder_path, program_file)

        if os.path.isfile(file_path):
            with open(file_path, "r", encoding="iso-8859-1") as f:
                # FIX 3: skip label-conflicting duplicates
                # ORIGINAL: java_programs.append(f.read())
                program_text = f.read()
            if label == 0 and program_text in pattern_texts:
                print(f"FIX 3: skipping {program_file}, identical to a file in pattern/ (label conflict)")
                continue
            java_programs.append(program_text)

            program_labels.append(label)
            program_files.append(program_file)


print(Counter(program_labels))

# Calculate embeddings for the Java programs line by line and take the mean
program_embeddings = []
for program in tqdm(java_programs, desc="Generating embeddings"):
    lines = program.split('\n')
    line_embeddings = [get_embeddings(line) for line in lines if line.strip()]
    program_embeddings.append(np.mean(line_embeddings, axis=0))

# Apply standard scaling to normalize the embeddings
scaler = StandardScaler()
normalized_embeddings = scaler.fit_transform(np.vstack(program_embeddings))

# Calculate the distance matrix based on the chosen metric
distance_matrix = calculate_distance_matrix(normalized_embeddings, metric=distance_metric)

# FIX 1: the block below created pseudo-labels with a leave-one-out kNN vote and then
# used them as the "true" labels for training and testing. It is no longer used; the
# real labels (program_labels) are used instead.
# ORIGINAL:
# k = 5  # Number of neighbors to consider
# knn_indices = np.argsort(distance_matrix)[:, 1:k+1]
# predicted_labels = []
# for indices in knn_indices:
#     neighbor_labels = [program_labels[i] for i in indices]
#     predicted_label = max(set(neighbor_labels), key=neighbor_labels.count)
#     predicted_labels.append(predicted_label)
# predicted_labels = np.array(predicted_labels)

# FIX 2: one unstratified 80/20 split (5-6 test files) replaced with stratified 5-fold
# cross-validation, so every file is predicted exactly once by a model that did not see it.
# ORIGINAL:
# X_train, X_test, y_train, y_test, files_train, files_test = train_test_split(
#     normalized_embeddings,
#     predicted_labels,
#     program_files,
#     test_size=0.2,
#     random_state=0
# )
# knn.fit(X_train, y_train)
# predictions = knn.predict(X_test)
y_test = np.array(program_labels)  # FIX 1: real labels
files_test = program_files

# Apply k-nearest neighbor classification
n_neighbors = 3  # Number of neighbors to consider
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=2137)  # FIX 2
predictions = cross_val_predict(knn, normalized_embeddings, y_test, cv=cv)  # FIX 2

# Calculate precision, recall, and f-score
precision = precision_score(y_test, predictions, average='weighted')
recall = recall_score(y_test, predictions, average='weighted')
f1 = f1_score(y_test, predictions, average='weighted')

# Print the classification results and performance metrics
# for i in range(len(X_test)):
#     print(f"File: {os.listdir(java_programs_folder)[i]}, Predicted Label: {predictions[i]}, True Label: {y_test[i]}")

for filename, prediction, true_label in zip(files_test, predictions, y_test):
    print(
        f"File: {filename}, "
        f"Predicted Label: {prediction}, "
        f"True Label: {true_label}"
    )
    
print(f"Precision: {precision}, Recall: {recall}, F-score: {f1}")



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14605.52it/s]


[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


FIX 3: skipping nons (14).java, identical to a file in pattern/ (label conflict)
Counter({1: 15, 0: 14})



Generating embeddings:   0%|          | 0/29 [00:00<?, ?it/s]

...


Generating embeddings:  93%|█████████▎| 27/29 [01:12<00:10,  5.14s/it]


Generating embeddings:  97%|█████████▋| 28/29 [01:13<00:03,  3.95s/it]


Generating embeddings: 100%|██████████| 29/29 [01:14<00:00,  2.95s/it]


Generating embeddings: 100%|██████████| 29/29 [01:14<00:00,  2.56s/it]

File: nons (1).java, Predicted Label: 0, True Label: 0
File: nons (2).java, Predicted Label: 0, True Label: 0
File: nons (3).java, Predicted Label: 0, True Label: 0
File: nons (4).java, Predicted Label: 1, True Label: 0
File: nons (5).java, Predicted Label: 0, True Label: 0
File: nons (6).java, Predicted Label: 0, True Label: 0
File: nons (7).java, Predicted Label: 0, True Label: 0
File: nons (8).java, Predicted Label: 0, True Label: 0
File: nons (9).java, Predicted Label: 1, True Label: 0
File: nons (10).java, Predicted Label: 0, True Label: 0
File: nons (11).java, Predicted Label: 0, True Label: 0
File: nons (12).java, Predicted Label: 0, True Label: 0
File: nons (13).java, Predicted Label: 0, True Label: 0
File: nons (15).java, Predicted Label: 0, True Label: 0
File: singleton (1).java, Predicted Label: 1, True Label: 1
File: singleton (2).java, Predicted Label: 1, True Label: 1
File: singleton (3).java, Predicted Label: 0, True Label: 1
File: singleton (4).java, Predicted Label: 1,